# 03 — Structure Prediction & Interface Analysis

This notebook folds the wild-type chain A sequence and the top candidate mutations from notebook 02 using **ESMFold**, via the free [ESM Atlas API](https://esmatlas.com/about#api) (`api.esmatlas.com`) — no GPU or local model install needed, since our 327-residue sequence is under its 400-residue limit. For each fold we check quality (**pLDDT**, the model's own per-residue confidence), and then cross-reference every candidate against the **real inter-subunit interface**, computed directly from the crystal tetramer downloaded in notebook 01 — not just the two literally-catalytic positions (Thr12/Thr89) flagged in notebook 02.

**Why go beyond Thr12/Thr89:** the active site caveat in `CLAUDE.md` is about more than two residues — the *whole* interface between subunits matters, since that's the biological context the crystal structure captures but a folded monomer alone cannot. ESMFold here predicts a single chain in isolation, so it can't tell us whether a mutation disrupts tetramer assembly — but the WT crystal structure already tells us exactly which chain A residues sit at that interface, and that's a fixed geometric fact we can check every candidate against directly, without needing to fold the whole tetramer per variant.

This notebook runs entirely on CPU with lightweight dependencies (`biopython`, `pandas`, and the standard library's `urllib` for the API calls) — no PyTorch, no Colab needed.

## Setup

In [1]:
import time
import urllib.request
import urllib.error
from pathlib import Path

import pandas as pd
from Bio.PDB import PDBParser, NeighborSearch
from Bio.PDB.Polypeptide import is_aa

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
STRUCTURES_DIR = RESULTS_DIR / "structures"
STRUCTURES_DIR.mkdir(parents=True, exist_ok=True)

## Load the reference sequence and top candidates from notebook 02

Same FASTA reader as notebook 02 (kept self-contained rather than importing across notebooks). We take the top `TOP_N` candidates by `esm_score`, already excluding substitutions that land directly on the catalytic residues (Thr12/Thr89) — notebook 02 flagged those in `at_catalytic_residue`.

In [2]:
def read_fasta(path: Path) -> tuple[str, str]:
    lines = Path(path).read_text().splitlines()
    header = lines[0].lstrip(">")
    sequence = "".join(lines[1:])
    return header, sequence


header, wt_sequence = read_fasta(DATA_DIR / "3eca_chainA.fasta")
print(header)
print(f"Reference sequence: {len(wt_sequence)} residues")

TOP_N = 8
scores_df = pd.read_csv(RESULTS_DIR / "esm2_mutation_scores.csv")
candidates = (
    scores_df[~scores_df["at_catalytic_residue"]]
    .sort_values("esm_score", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)
print(f"Top {TOP_N} candidates:")
candidates[["mutation", "position", "wt", "mut", "esm_score"]]

3ECA_A|E.coli_L-asparaginase_II|327aa
Reference sequence: 327 residues
Top 8 candidates:


,mutation,position,wt,mut,esm_score
0,L1M,1,L,M,9.322736
1,C105S,105,C,S,6.366481
2,Y250N,250,Y,N,5.661932
3,Y250S,250,Y,S,4.890075
4,A266V,266,A,V,4.445523
5,A266I,266,A,I,3.880548
6,Y250P,250,Y,P,3.873255
7,A228V,228,A,V,3.486542


## Build variant sequences

Apply each candidate substitution to the wild-type sequence. The assertion re-confirms the wild-type residue at that position matches what notebook 02 recorded — the same "don't silently trust upstream data" habit from notebook 01.

In [3]:
def apply_mutation(sequence: str, position: int, wt_aa: str, mut_aa: str) -> str:
    idx = position - 1  # positions are 1-indexed, matching PDB numbering
    assert sequence[idx] == wt_aa, f"Expected {wt_aa} at position {position}, found {sequence[idx]}"
    return sequence[:idx] + mut_aa + sequence[idx + 1:]


sequences_to_fold = [("WT", wt_sequence)]
for row in candidates.itertuples():
    variant_seq = apply_mutation(wt_sequence, row.position, row.wt, row.mut)
    sequences_to_fold.append((row.mutation, variant_seq))

print(f"{len(sequences_to_fold)} sequences queued for folding (1 WT + {TOP_N} variants)")

9 sequences queued for folding (1 WT + 8 variants)


## Fold each sequence with ESMFold

The free API occasionally returns a transient `504 Gateway Timeout` under load (observed during development) even though the request eventually succeeds on retry — so we retry a few times with a generous per-request timeout rather than treating one timeout as a hard failure. A short pause between requests is polite to a shared free service.

In [4]:
FOLD_URL = "https://api.esmatlas.com/foldSequence/v1/pdb/"


def fold_sequence(sequence: str, timeout: int = 180, retries: int = 3) -> str:
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(FOLD_URL, data=sequence.encode(), method="POST")
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read().decode()
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as exc:
            print(f"  attempt {attempt}/{retries} failed ({exc}), retrying...")
            time.sleep(5)
    raise RuntimeError(f"Folding failed after {retries} attempts")


pdb_paths = {}
for label, seq in sequences_to_fold:
    out_path = STRUCTURES_DIR / f"{label}.pdb"
    if out_path.exists():
        print(f"{label}: already folded, skipping")
    else:
        print(f"Folding {label}...")
        t0 = time.time()
        pdb_text = fold_sequence(seq)
        out_path.write_text(pdb_text)
        print(f"  done in {time.time() - t0:.1f}s")
        time.sleep(1)
    pdb_paths[label] = out_path

print("All structures folded.")

Folding WT...


  done in 0.8s


Folding L1M...


  done in 15.7s


Folding C105S...


...

Folding Y250P...


  done in 15.1s


Folding A228V...


  done in 20.5s


All structures folded.


## Extract per-residue pLDDT

pLDDT (predicted Local Distance Difference Test) is ESMFold's own per-residue confidence in its prediction, stored in the PDB's B-factor column. It's reported per atom in this API's output (not uniformly per residue), so by convention we read it off the **CA atom** specifically. The API returns values on a 0-1 scale; we rescale by 100 to match the usual 0-100 pLDDT convention (>90 very high confidence, 70-90 confident, 50-70 low, <50 very low).

In [5]:
parser = PDBParser(QUIET=True)


def get_plddt_by_position(pdb_path: Path) -> dict[int, float]:
    structure = parser.get_structure(pdb_path.stem, pdb_path)
    chain = next(structure[0].get_chains())
    residues = [r for r in chain if is_aa(r, standard=True)]
    return {r.id[1]: r["CA"].get_bfactor() * 100 for r in residues}


plddt_by_label = {label: get_plddt_by_position(path) for label, path in pdb_paths.items()}

wt_plddt = plddt_by_label["WT"]
print(f"WT mean pLDDT: {sum(wt_plddt.values()) / len(wt_plddt):.1f}")

WT mean pLDDT: 91.6


## Real inter-subunit interface, from the crystal tetramer

We use the WT tetramer structure downloaded in notebook 01 (`data/pdb/pdb3eca.ent`) to find every chain A residue with an atom within 5 A of any atom in chains B, C, or D. This is a property of the wild-type assembly's geometry, computed once — it doesn't need to be redone per variant.

In [6]:
tetramer = parser.get_structure("3ECA", DATA_DIR / "pdb" / "pdb3eca.ent")
tetramer_model = tetramer[0]

other_chain_atoms = [a for c in tetramer_model if c.id != "A" for a in c.get_atoms()]
neighbor_search = NeighborSearch(other_chain_atoms)

INTERFACE_CUTOFF = 5.0  # Angstrom
interface_positions = set()
for res in tetramer_model["A"]:
    if not is_aa(res, standard=True):
        continue
    if any(neighbor_search.search(atom.coord, INTERFACE_CUTOFF) for atom in res):
        interface_positions.add(res.id[1])

print(f"{len(interface_positions)} of {len(wt_plddt)} chain A residues "
      f"({len(interface_positions) / len(wt_plddt):.0%}) sit at the inter-subunit interface")
print("12 in interface:", 12 in interface_positions)
print("89 in interface:", 89 in interface_positions)

112 of 327 chain A residues (34%) sit at the inter-subunit interface
12 in interface: True
89 in interface: True


## Combine into a ranking table

For each candidate: the ESM-2 score (notebook 02), the folded variant's mean pLDDT, the pLDDT specifically at the mutated position (and how it compares to the same position in the WT fold), whether the position sits at the real inter-subunit interface, and whether it's within 3 residues of either sequence terminus — N/C-terminal positions are structurally less constrained and ESM-2 tends to over-score them (notice `L1M` topping notebook 02's ranking; that's likely this effect, not a genuine stability signal).

In [7]:
TERMINUS_MARGIN = 3
seq_len = len(wt_sequence)

rows = []
for row in candidates.itertuples():
    label = row.mutation
    variant_plddt = plddt_by_label[label]
    rows.append({
        "mutation": label,
        "position": row.position,
        "esm_score": row.esm_score,
        "mean_plddt": sum(variant_plddt.values()) / len(variant_plddt),
        "plddt_at_position": variant_plddt[row.position],
        "wt_plddt_at_position": wt_plddt[row.position],
        "plddt_delta_at_position": variant_plddt[row.position] - wt_plddt[row.position],
        "at_subunit_interface": row.position in interface_positions,
        "near_terminus": row.position <= TERMINUS_MARGIN or row.position > seq_len - TERMINUS_MARGIN,
    })

ranking_df = pd.DataFrame(rows).sort_values("esm_score", ascending=False).reset_index(drop=True)
ranking_df

,mutation,position,esm_score,mean_plddt,plddt_at_position,wt_plddt_at_position,plddt_delta_at_position,at_subunit_interface,near_terminus
0,L1M,1,9.322736,90.593272,89.0,91.0,-2.0,False,True
1,C105S,105,6.366481,91.782875,90.0,94.0,-4.0,False,False
2,Y250N,250,5.661932,91.464832,91.0,94.0,-3.0,True,False
3,Y250S,250,4.890075,91.532110,90.0,94.0,-4.0,True,False
4,A266V,266,4.445523,91.633028,97.0,97.0,0.0,False,False
5,A266I,266,3.880548,91.642202,97.0,97.0,0.0,False,False
6,Y250P,250,3.873255,91.455657,92.0,94.0,-2.0,True,False
7,A228V,228,3.486542,91.538226,97.0,97.0,0.0,False,False


In [8]:
out_path = RESULTS_DIR / "structure_scores.csv"
ranking_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to ..\results\structure_scores.csv


## Summary

- Folded the wild type and top 8 ESM-2-ranked candidates (catalytic-residue substitutions already excluded) using ESMFold via the free ESM Atlas API — CPU-only, no Colab needed for this notebook.
- Computed each fold's per-residue pLDDT confidence and compared it at the mutated position against the wild-type fold.
- Computed the *real* inter-subunit interface from the crystal tetramer (112/327 chain A residues, ~34%, sit within 5 A of another subunit) and flagged every candidate against it — a much broader check than the two literal catalytic positions.
- Flagged candidates sitting near either sequence terminus, since those are prone to inflated ESM-2 scores from reduced structural constraint rather than genuine evolutionary tolerance.
- Saved the combined table to `results/structure_scores.csv`.
- **Caveat:** ESMFold here predicts each chain as an isolated monomer — a high pLDDT means "this sequence folds into a well-defined structure by itself," not "the tetramer still assembles correctly." The interface flag is how we compensate for that blind spot with the one thing we can check directly from real geometry.
- **Next:** `04_nnp_stability_scoring.ipynb` — relax each folded structure with a neural network potential (MACE) and estimate a ΔΔG-like stability proxy, to combine with `esm_score` and `mean_plddt` into a final ranking.